In [8]:
import json, os

kaggle_json = {
    "username": "naitiiik31",
    "key": "YOUR_KAGGLE_TOKEN"
}

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_json, f)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("kaggle.json created")


kaggle.json created


In [9]:
!kaggle datasets list | head


ref                                                                title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------------  -------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
neurocipher/heartdisease                                           Heart Disease                                            3491  2025-12-11 15:29:14.327000           2114        332  1.0              
saidaminsaidaxmadov/chocolate-sales                                Chocolate Sales                                        468320  2026-01-04 14:23:35.490000              0         65  1.0              
rockyt07/social-media-user-analysis                                Social Media User Analysis                          247842357  2026-01-14 02:28:41.970000              0         76  1.0     

In [10]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
!unzip imdb-dataset-of-50k-movie-reviews.zip


Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
imdb-dataset-of-50k-movie-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  imdb-dataset-of-50k-movie-reviews.zip
replace IMDB Dataset.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: IMDB Dataset.csv        


In [11]:
!ls


'IMDB Dataset.csv'			 movies_metadata.csv
 imdb-dataset-of-50k-movie-reviews.zip	 sample_data


In [12]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [13]:
import re

def clean_review(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text

df["clean_review"] = df["review"].apply(clean_review)


In [14]:
movies_df = pd.read_csv("movies_metadata.csv", low_memory=False)
movies_df = movies_df[["title"]].dropna()
movies_df.head()


,title
0,Toy Story
1,Jumanji
2,Grumpier Old Men
3,Waiting to Exhale
4,Father of the Bride Part II


In [15]:
def clean_title(title):
    title = title.lower()
    title = re.sub(r"[^a-z0-9\s]", "", title)
    return title.strip()

movies_df["clean_title"] = movies_df["title"].apply(clean_title)


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X = vectorizer.fit_transform(df["clean_review"])
y = df["sentiment"]


In [17]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

print("Sentiment model trained")


Sentiment model trained


In [18]:
def get_movie_sentiment(movie_title, df, model, vectorizer):
    movie_title = clean_title(movie_title)

    if len(movie_title.split()) < 2:
        return None

    matched = df[df["clean_review"].str.contains(movie_title)]

    if matched.empty:
        return None

    X_movie = vectorizer.transform(matched["clean_review"])
    preds = model.predict(X_movie)

    matched = matched.copy()
    matched["predicted_sentiment"] = preds

    return {
        "total_positive": int((preds == "positive").sum()),
        "total_negative": int((preds == "negative").sum()),
        "top_positive_reviews": matched[
            matched["predicted_sentiment"] == "positive"
        ]["review"].head(5).tolist(),
        "top_negative_reviews": matched[
            matched["predicted_sentiment"] == "negative"
        ]["review"].head(5).tolist()
    }


In [19]:
movie_name = "Toy Story"
print("Movie:", movie_name)

result = get_movie_sentiment(movie_name, df, model, vectorizer)

if result:
    print("Positive:", result["total_positive"])
    print("Negative:", result["total_negative"])

    print("\nTop 5 Positive Reviews:")
    for i, r in enumerate(result["top_positive_reviews"], 1):
        print(f"{i}. {r[:300]}")

    print("\nTop 5 Negative Reviews:")
    for i, r in enumerate(result["top_negative_reviews"], 1):
        print(f"{i}. {r[:300]}")
else:
    print("No reviews found")


Movie: Toy Story
Positive: 31
Negative: 12

Top 5 Positive Reviews:
1. Such a joyous world has been created for us in Pixar's A Bug's Life; we're immersed in a universe which could only be documented this enjoyably on film, but more precisely a universe which could only be documented through the world of animation. For those who have forgotten what a plentiful and exub
2. The movie "Atlantis: The Lost Empire" is a shining gem in the rubble of films produced by the Disney Studios recently. Parents who have had to sit through "The Jungle Book 2" or even a Pokemon movie will surely appreciate this one.<br /><br />The film is one of few to attempt at an original story; p
3. This is the kind of picture John Lassiter would be making today, if it weren't for advances in CGI. And that's just to say that he'd be forgotten, too, if technology hadn't made things sexy and kewl since 1983. _Twice..._ has got the same wit, imagination, and sense of real excitement that you'd fin
4. I was so eager to

In [20]:
movie_sentiment_cache = {}

for title in movies_df["title"]:
    data = get_movie_sentiment(title, df, model, vectorizer)
    if data:
        movie_sentiment_cache[title] = data

len(movie_sentiment_cache)


12198

In [21]:
import pickle

pickle.dump(movie_sentiment_cache, open("movie_sentiment.pkl", "wb"))
pickle.dump(model, open("sentiment_model.pkl", "wb"))
pickle.dump(vectorizer, open("sentiment_vectorizer.pkl", "wb"))

print("All PKL files saved")


All PKL files saved
